# 03 – Data Transformation & Feature Engineering

This notebook covers:

**Part A – Products**
- Filter to Makeup category
- Missing-value imputation
- Feature engineering (price, brand, popularity, ingredients)
- Numerical scaling & one-hot encoding
- Assemble `df_product_final`

**Part B – Reviews**
- Text cleaning & normalisation
- TF-IDF vectorisation
- VADER sentiment scoring
- Assemble `reviews_tfidf_df`

**Part C – Save to S3**
- Upload both feature matrices to `s3://ads508-team1-sephora/Model/final_features/`

**Input:** `s3://ads508-team1-sephora/curated/`
**Output:** `s3://ads508-team1-sephora/Model/final_features/`

## 1. Imports

In [34]:
import os
import shutil
import re
import warnings
warnings.filterwarnings("ignore")

import boto3
import pandas as pd
import numpy as np
import nltk
import sklearn

from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# nltk.download('stopwords')  # Uncomment on first run

## 2. Load Curated Datasets

In [35]:
BUCKET       = "ads508-team1-sephora"
S3_PRODUCTS  = f"s3://{BUCKET}/curated/products/products.parquet"
S3_REVIEWS   = f"s3://{BUCKET}/curated/reviews/reviews.parquet"

df_products = pd.read_parquet(S3_PRODUCTS)
df_reviews  = pd.read_parquet(S3_REVIEWS)

print(f"Products (all categories) : {df_products.shape}")
print(f"Reviews                   : {df_reviews.shape}")

Products (all categories) : (8494, 27)
Reviews                   : (1094411, 18)


---
# Part A – Product Feature Engineering

Steps covered:
1. Filter to Makeup category
2. Missing-value imputation
3. Feature engineering (price, brand, popularity, ingredients)
4. Numerical scaling & one-hot encoding
5. Assemble `df_product_final`

## A1. Filter to Makeup Category

In [36]:
df_products = (
    df_products[df_products["primary_category"] == "Makeup"]
    .copy()
    .reset_index(drop=True)
)

print(f"Makeup products shape : {df_products.shape}")
print("\nSub-category breakdown:")
print(df_products["secondary_category"].value_counts())

Makeup products shape : (2369, 27)

Sub-category breakdown:
secondary_category
Eye                      711
Face                     659
Lip                      411
Brushes & Applicators    233
Cheek                    165
Nail                      52
Accessories               45
Mini Size                 37
Value & Gift Sets         36
Makeup Palettes           20
Name: count, dtype: int64


## A2. Missing-Value Imputation

| Column(s) | Strategy | Rationale |
|-----------|----------|-----------|
| `rating` | Mean fill | Small % missing; mean is a neutral substitute |
| `reviews` | Fill with 0 | No reviews = 0 count |
| Text / category columns | Fill with `"NA"` | Preserves column for downstream text handling |
| `value_price_usd`, `sale_price_usd` | Fill with `price_usd` | No discount data → assume regular price |
| `child_max_price`, `child_min_price` | Fill with 0 | No child variants → range is 0 |

In [37]:
# 1. Rating – mean imputation
df_products["rating"] = df_products["rating"].fillna(df_products["rating"].mean())

# 2. Reviews – zero fill
df_products["reviews"] = df_products["reviews"].fillna(0)

# 3. Text / categorical columns
TEXT_COLS = [
    "size", "variation_type", "variation_value", "variation_desc",
    "ingredients", "highlights", "secondary_category", "tertiary_category",
]
df_products[TEXT_COLS] = df_products[TEXT_COLS].fillna("NA")

# 4. Price columns
df_products["value_price_usd"] = df_products["value_price_usd"].fillna(df_products["price_usd"])
df_products["sale_price_usd"]  = df_products["sale_price_usd"].fillna(df_products["price_usd"])

# 5. Child price range
df_products["child_max_price"] = df_products["child_max_price"].fillna(0)
df_products["child_min_price"] = df_products["child_min_price"].fillna(0)

# Confirm no remaining nulls
assert df_products.isnull().sum().sum() == 0, "Remaining nulls found!"
print("All missing values resolved.")
print(df_products.isnull().mean().mul(100).round(2).to_string())

All missing values resolved.
product_id            0.0
product_name          0.0
brand_id              0.0
brand_name            0.0
loves_count           0.0
rating                0.0
reviews               0.0
size                  0.0
variation_type        0.0
variation_value       0.0
variation_desc        0.0
ingredients           0.0
price_usd             0.0
value_price_usd       0.0
sale_price_usd        0.0
limited_edition       0.0
new                   0.0
online_only           0.0
out_of_stock          0.0
sephora_exclusive     0.0
highlights            0.0
primary_category      0.0
secondary_category    0.0
tertiary_category     0.0
child_count           0.0
child_max_price       0.0
child_min_price       0.0


## A3. Feature Engineering

### A3.1 Price Features

In [38]:
df_products["price_log"]    = np.log1p(df_products["price_usd"])
df_products["is_on_sale"]   = (df_products["sale_price_usd"] < df_products["price_usd"]).astype(int)
df_products["discount_pct"] = (
    (df_products["price_usd"] - df_products["sale_price_usd"]) / df_products["price_usd"]
).clip(lower=0)
df_products["value_ratio"]  = df_products["value_price_usd"] / df_products["price_usd"]

df_products[["price_usd", "price_log", "is_on_sale", "discount_pct", "value_ratio"]].head()

,price_usd,price_log,is_on_sale,discount_pct,value_ratio
0,35.0,3.583519,0,0.0,1.0
1,20.0,3.044522,0,0.0,1.0
2,32.0,3.496508,0,0.0,1.0
3,19.0,2.995732,0,0.0,1.0
4,22.0,3.135494,0,0.0,1.0


### A3.2 Price Bucketing

In [39]:
BINS   = [0, 25, 50, 100, np.inf]
LABELS = ["budget", "low_mid", "mid", "premium"]

df_products["price_bucket"] = pd.cut(df_products["price_usd"], bins=BINS, labels=LABELS)

print(df_products["price_bucket"].value_counts())

price_bucket
low_mid    1239
budget      895
mid         212
premium      23
Name: count, dtype: int64


### A3.3 Brand Grouping (Top-20 → Other)

In [40]:
top_brands = df_products["brand_name"].value_counts().nlargest(20).index
df_products["brand_group"] = df_products["brand_name"].where(
    df_products["brand_name"].isin(top_brands), other="Other"
)

print(df_products["brand_group"].value_counts())

brand_group
Other                      1082
SEPHORA COLLECTION          188
tarte                       100
Anastasia Beverly Hills      88
Charlotte Tilbury            76
Fenty Beauty by Rihanna      74
Hourglass                    67
MAKE UP FOR EVER             67
NARS                         59
CLINIQUE                     59
Dior                         56
HUDA BEAUTY                  54
Too Faced                    53
Urban Decay                  50
Benefit Cosmetics            49
PAT McGRATH LABS             49
Laura Mercier                47
Bobbi Brown                  39
Natasha Denona               38
Smashbox                     38
MILK MAKEUP                  36
Name: count, dtype: int64


### A3.4 Popularity & Engagement Features

In [41]:
df_products["log_loves"]        = np.log1p(df_products["loves_count"])
df_products["loves_per_price"]  = df_products["loves_count"] / df_products["price_usd"]
df_products["popularity_index"] = df_products["reviews"] * df_products["rating"]

df_products[["product_name", "loves_count", "log_loves", "loves_per_price", "popularity_index"]]    .sort_values("loves_count", ascending=False).head()

,product_name,loves_count,log_loves,loves_per_price,popularity_index
1645,Soft Pinch Liquid Blush,1401068,14.152746,60916.000000,21466.9948
1404,Radiant Creamy Concealer,1153594,13.958394,36049.812500,55517.1960
1742,Cream Lip Stain Liquid Lipstick,1029051,13.844149,68603.400000,48000.6311
550,Gloss Bomb Universal Lip Luminizer,968317,13.783316,46110.333333,56258.8552
551,Pro Filt’r Soft Matte Longwear Liquid Foundation,856497,13.660607,21412.425000,68342.8860


### A3.5 Ingredient TF-IDF Features

In [42]:
INGREDIENT_MAP = {
    "aqua": "water", "eau": "water", "water/aqua": "water",
    "aqua/water": "water", "aqua/water/eau": "water", "water/aqua/eau": "water",
    "tocopherol": "vitamin_e", "tocopheryl acetate": "vitamin_e",
    "sodium hyaluronate": "hyaluronic_acid",
    "ci 77891": "titanium_dioxide", "titanium dioxide/ci 77891": "titanium_dioxide",
    "ci 77891/titanium dioxide": "titanium_dioxide",
    "ci 77491": "iron_oxides", "ci 77492": "iron_oxides", "ci 77499": "iron_oxides",
    "fragrance": "fragrance", "parfum": "fragrance", "fragrance/parfum": "fragrance",
}

def clean_ingredient(raw: str) -> str:
    s = (
        raw.lower().strip()
        .replace("[", "").replace("]", "").replace("'", "")
        .replace("+/-", "").strip()
    )
    return INGREDIENT_MAP.get(s, s)

ingredients_long = (
    df_products[["ingredients"]]
    .replace("NA", np.nan)
    .dropna()
    .assign(ingredient=lambda x: x["ingredients"].str.split(","))
    .explode("ingredient")
    .copy()
)
ingredients_long["product_idx"]      = ingredients_long.index
ingredients_long["ingredient_clean"] = ingredients_long["ingredient"].apply(clean_ingredient)

# Count ingredients per product
df_products["num_ingredients"] = (
    ingredients_long.groupby("product_idx")["ingredient_clean"].count()
    .reindex(df_products.index, fill_value=0)
)

# Join cleaned ingredient strings per product (for vectorisation)
ingredients_per_product = (
    ingredients_long.groupby("product_idx")["ingredient_clean"]
    .apply(lambda x: ",".join(x))
)
df_products["ingredients_clean"] = (
    ingredients_per_product.reindex(df_products.index).fillna("")
)

# TF-IDF vectorisation
vectorizer_ing = TfidfVectorizer(
    token_pattern=r"[^,]+",
    max_features=200,
    min_df=5,
    max_df=0.8,
)
X_ingredients     = vectorizer_ing.fit_transform(df_products["ingredients_clean"])
ingredients_tfidf = pd.DataFrame.sparse.from_spmatrix(
    X_ingredients,
    columns=[f"ing_{c}" for c in vectorizer_ing.get_feature_names_out()],
    index=df_products.index,
)

print(f"Ingredient TF-IDF shape : {ingredients_tfidf.shape}")
ingredients_tfidf.head()

Ingredient TF-IDF shape : (2369, 200)


,ing_1,ing_2-hexanediol,ing_acacia senegal gum,ing_acrylates copolymer,ing_alcohol,ing_alcohol denat.,ing_alumina,ing_aluminum hydroxide,ing_aqua (water),ing_ascorbyl palmitate,...,ing_vp/hexadecene copolymer,ing_water,ing_xanthan gum,ing_yellow 5 lake (ci 19140),ing_yellow 5 lake (ci 19140).,ing_yellow 5 lake),ing_yellow 6 lake (ci 15985),ing_zea mays (corn) starch,ing_zinc myristate,ing_zinc stearate
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0.428037,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0.169796,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0.157461,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## A4. Feature Transformation

### A4.1 Numerical Scaling (StandardScaler)

In [43]:
COLS_TO_SCALE = [
    "price_log", "discount_pct", "value_ratio", "log_loves",
    "child_count", "child_max_price", "child_min_price",
    "num_ingredients", "popularity_index", "loves_per_price",
]
# Keep only columns that actually exist after engineering
COLS_TO_SCALE = [c for c in COLS_TO_SCALE if c in df_products.columns]

scaler = StandardScaler()
df_products[COLS_TO_SCALE] = scaler.fit_transform(df_products[COLS_TO_SCALE])

print(f"Scaled {len(COLS_TO_SCALE)} columns: {COLS_TO_SCALE}")

Scaled 10 columns: ['price_log', 'discount_pct', 'value_ratio', 'log_loves', 'child_count', 'child_max_price', 'child_min_price', 'num_ingredients', 'popularity_index', 'loves_per_price']


### A4.2 One-Hot Encoding of Categorical Columns

In [44]:
CATEGORICAL_COLS = ["secondary_category", "tertiary_category", "price_bucket", "brand_group"]

df_products = pd.get_dummies(
    df_products,
    columns=CATEGORICAL_COLS,
    drop_first=True,   # avoids multicollinearity
    dtype=int,
)

print(f"Shape after OHE : {df_products.shape}")

Shape after OHE : (2369, 107)


## A5. Assemble `df_product_final`

In [45]:
DROP_COLS = [
    "product_id", "product_name", "brand_id", "brand_name",
    "price_usd", "value_price_usd", "sale_price_usd",
    "loves_count", "rating", "reviews",
    "log_loves", "loves_per_price", "popularity_index",
    "size", "variation_type", "variation_value", "variation_desc",
    "ingredients", "ingredients_clean", "highlights",
]
# Only drop columns that actually exist
DROP_COLS = [c for c in DROP_COLS if c in df_products.columns]

df_product_final = pd.concat(
    [df_products.drop(columns=DROP_COLS), ingredients_tfidf],
    axis=1,
)

print(f"Final product feature matrix : {df_product_final.shape}")
print(f"Columns (first 20): {list(df_product_final.columns[:20])}")
df_product_final.head()

Final product feature matrix : (2369, 287)
Columns (first 20): ['limited_edition', 'new', 'online_only', 'out_of_stock', 'sephora_exclusive', 'primary_category', 'child_count', 'child_max_price', 'child_min_price', 'price_log', 'is_on_sale', 'discount_pct', 'value_ratio', 'num_ingredients', 'secondary_category_Brushes & Applicators', 'secondary_category_Cheek', 'secondary_category_Eye', 'secondary_category_Face', 'secondary_category_Lip', 'secondary_category_Makeup Palettes']


,limited_edition,new,online_only,out_of_stock,sephora_exclusive,primary_category,child_count,child_max_price,child_min_price,price_log,...,ing_vp/hexadecene copolymer,ing_water,ing_xanthan gum,ing_yellow 5 lake (ci 19140),ing_yellow 5 lake (ci 19140).,ing_yellow 5 lake),ing_yellow 6 lake (ci 15985),ing_zea mays (corn) starch,ing_zinc myristate,ing_zinc stearate
0,0,0,1,0,0,Makeup,-0.512608,-0.900507,-0.877163,0.379117,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,1,Makeup,-0.187795,0.177735,0.246876,-0.787481,...,0.428037,0,0,0,0,0,0,0,0,0
2,0,0,0,0,1,Makeup,0.028747,0.824680,0.921300,0.190790,...,0,0.169796,0,0,0,0,0,0,0,0
3,0,0,0,0,1,Makeup,-0.512608,-0.900507,-0.877163,-0.893082,...,0,0.157461,0,0,0,0,0,0,0,0
4,0,0,1,0,1,Makeup,-0.296066,0.285559,0.359280,-0.590583,...,0,0,0,0,0,0,0,0,0,0


---
# Part B – Review Feature Engineering

Steps covered:
1. Text cleaning & normalisation
2. TF-IDF vectorisation on review text
3. VADER sentiment scoring
4. Assemble `reviews_tfidf_df`

## B1. Text Cleaning & TF-IDF Vectorisation

In [46]:
stop_words = set(nltk.corpus.stopwords.words("english"))
tokenizer  = nltk.WordPunctTokenizer()

def normalize_text(doc: str) -> str:
    doc    = re.sub(r"[^a-zA-Z0-9\s]", "", str(doc), flags=re.I)
    tokens = tokenizer.tokenize(doc.lower().strip())
    return " ".join(t for t in tokens if t not in stop_words)

# Sample for speed; increase or remove .sample() for full corpus
SAMPLE_SIZE = 10_000
df_reviews_sample = df_reviews.sample(SAMPLE_SIZE, random_state=42).copy()
df_reviews_sample["review_text_clean"] = (
    df_reviews_sample["review_text"].fillna("").apply(normalize_text)
)

vectorizer_rev = TfidfVectorizer(
    max_features=3000,
    min_df=5,
    max_df=0.8,
    ngram_range=(1, 2),
)
X_reviews     = vectorizer_rev.fit_transform(df_reviews_sample["review_text_clean"])
reviews_tfidf = pd.DataFrame.sparse.from_spmatrix(
    X_reviews,
    columns=[f"rev_{c}" for c in vectorizer_rev.get_feature_names_out()],
    index=df_reviews_sample.index,
)

reviews_tfidf_df = pd.concat(
    [df_reviews_sample[["rating"]].reset_index(drop=True),
     reviews_tfidf.reset_index(drop=True)],
    axis=1,
)

print(f"Review TF-IDF shape : {reviews_tfidf_df.shape}")

Review TF-IDF shape : (10000, 3001)


## B2. VADER Sentiment Scoring

In [47]:
analyzer = SentimentIntensityAnalyzer()

def get_vader_scores(text: str) -> pd.Series:
    text = str(text).strip()
    if not text:
        return pd.Series({"neg": None, "neu": None, "pos": None, "compound": None})
    s = analyzer.polarity_scores(text)
    return pd.Series({"neg": s["neg"], "neu": s["neu"], "pos": s["pos"], "compound": s["compound"]})

vader_cols       = df_reviews_sample["review_text"].apply(get_vader_scores)
df_reviews_vader = pd.concat(
    [df_reviews_sample[["rating", "review_text"]], vader_cols], axis=1
)

print(df_reviews_vader.shape)
df_reviews_vader.head()

(10000, 6)


,rating,review_text,neg,neu,pos,compound
1049894,1,I have used 2 bottles of this product and it d...,0.000,0.891,0.109,0.3252
619554,5,"I love this product, I feel like it really cal...",0.000,0.606,0.394,0.9230
634782,5,Lately I’ve been loving my skin and this toner...,0.000,0.906,0.094,0.7193
706815,5,This toner is one of the best purchases I have...,0.000,0.863,0.137,0.9243
731656,5,I have been loving all the products so far the...,0.044,0.675,0.281,0.9238


## B3. Assemble `reviews_tfidf_df`

In [48]:
# Append compound sentiment score to the TF-IDF matrix
reviews_tfidf_df["compound_score"] = df_reviews_vader["compound"].values

# Scale the sentiment score
scaler_sent = StandardScaler()
reviews_tfidf_df[["compound_score"]] = scaler_sent.fit_transform(
    reviews_tfidf_df[["compound_score"]]
)

print(f"Final reviews_tfidf_df shape : {reviews_tfidf_df.shape}")
reviews_tfidf_df.head()

Final reviews_tfidf_df shape : (10000, 3002)


,rating,rev_10,rev_10 minutes,rev_10 years,rev_100,...,rev_yttp,rev_zero,rev_zits,rev_zone,compound_score
0,1,0,0,0,0,...,0,0,0,0,-0.830245
1,5,0,0,0,0,...,0,0,0,0,0.593412
2,5,0,0,0,0,...,0,0,0,0,0.108302
3,5,0,0,0,0,...,0,0,0,0,0.596508
4,5,0,0,0,0,...,0,0,0,0,0.595317


---
# Part C – Save Feature Matrices to S3

Both final feature matrices are saved to:
```
s3://ads508-team1-sephora/Model/final_features/
```
This path is the input for `04_data_preparation.ipynb`.

## C1. Upload to S3

In [49]:
S3_FINAL_FEATURES = "Model/final_features"
LOCAL_TMP         = "./temp_features"

os.makedirs(LOCAL_TMP, exist_ok=True)
s3_client = boto3.client("s3")

feature_files = {
    "df_product_final": df_product_final,
    "reviews_tfidf_df": reviews_tfidf_df,
}

for name, df in feature_files.items():
    local_path = f"{LOCAL_TMP}/{name}.parquet"
    s3_key     = f"{S3_FINAL_FEATURES}/{name}.parquet"

    # Densify any sparse columns before writing to Parquet
    # (TF-IDF matrices are stored as sparse; PyArrow requires dense)
    df_dense = df.copy()
    sparse_cols = [
        col for col in df_dense.columns
        if hasattr(df_dense[col], "sparse")
    ]
    if sparse_cols:
        df_dense[sparse_cols] = df_dense[sparse_cols].astype(float)
        print(f"  Densified {len(sparse_cols)} sparse columns in {name}")

    df_dense.to_parquet(local_path, index=False, engine="pyarrow")
    s3_client.upload_file(local_path, BUCKET, s3_key)
    print(f"Uploaded  s3://{BUCKET}/{s3_key}  {df_dense.shape}")

shutil.rmtree(LOCAL_TMP)
print("\nAll feature matrices saved and temp directory cleaned up.")


  Densified 200 sparse columns in df_product_final
Uploaded  s3://ads508-team1-sephora/Model/final_features/df_product_final.parquet  (2369, 287)
  Densified 3000 sparse columns in reviews_tfidf_df
Uploaded  s3://ads508-team1-sephora/Model/final_features/reviews_tfidf_df.parquet  (10000, 3002)

All feature matrices saved and temp directory cleaned up.


## C2. Verification

In [50]:
# Reload and confirm shapes match
for name, expected_df in feature_files.items():
    s3_path   = f"s3://{BUCKET}/{S3_FINAL_FEATURES}/{name}.parquet"
    reloaded  = pd.read_parquet(s3_path)
    status    = "✓" if reloaded.shape == expected_df.shape else "✗ MISMATCH"
    print(f"{name:25s}  expected={expected_df.shape}  reloaded={reloaded.shape}  {status}")

df_product_final           expected=(2369, 287)  reloaded=(2369, 287)  ✓
reviews_tfidf_df           expected=(10000, 3002)  reloaded=(10000, 3002)  ✓


---
## Summary

| Part | Output | Shape | S3 Path |
|------|--------|-------|---------|
| A – Products | `df_product_final` | (n_products, ~250+) | `Model/final_features/df_product_final.parquet` |
| B – Reviews | `reviews_tfidf_df` | (10k, ~3002) | `Model/final_features/reviews_tfidf_df.parquet` |

➡ Next notebook: `04_data_preparation.ipynb`